# 03 — Gráficas históricas de features y pronóstico 2026

Genera una figura independiente para cada feature numérica evaluada y para cada variable categórica incluida en las representaciones. Finaliza con los cuatro paneles de pronóstico.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import matplotlib.pyplot as plt
import pandas as pd

try:
    from google.colab import drive
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    drive.mount('/content/drive')
    REPO_ROOT = Path('/content/suelosabio')
    if not (REPO_ROOT / '.git').exists():
        subprocess.run([
            'git', 'clone', '--depth', '1', '--branch', 'feature/SCRUM-17',
            'https://github.com/cybercolombia/suelosabio.git', str(REPO_ROOT)
        ], check=True)
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q', '-r',
        str(REPO_ROOT / 'notebooks/CropForecasting/requirements.txt')
    ], check=True)
else:
    REPO_ROOT = Path.cwd()
    while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / '.git').exists():
        REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT))

In [ ]:
from notebooks.CropForecasting.config import load_forecasting_config
from notebooks.CropForecasting.visualization import (
    EMBEDDING_COLUMNS,
    model_feature_inventory,
    plot_categorical_encoding,
    plot_forecast_panels,
    plot_numeric_feature_history,
)

CONFIG = load_forecasting_config(in_colab=EN_COLAB, mount_drive=False)
dataset = pd.read_parquet(CONFIG.dataset_root / 'dataset_definitivo.parquet')
pronostico = pd.read_parquet(CONFIG.model_root / 'pronostico_2026.parquet')
manifest = json.loads((CONFIG.dataset_root / 'manifest.json').read_text(encoding='utf-8'))
inventory = model_feature_inventory(manifest['climate_features'])
display(inventory)

## Comportamiento histórico de cada feature

Cada figura muestra mediana y rango intercuartílico por año, separando semestres A y B. El modelo final utiliza `rendimiento_lag_1`; el inventario adicional documenta todas las variables comparadas durante la selección.

In [ ]:
GRAFICAR_FEATURES_NUMERICAS = True

if GRAFICAR_FEATURES_NUMERICAS:
    numericas = inventory[inventory.tipo.eq('NUMERICA')].variable.drop_duplicates()
    for feature in numericas:
        figure = plot_numeric_feature_history(dataset, feature)
        display(figure)
        plt.close(figure)

## Variables categóricas incluidas en las representaciones

`one_hot_entity` usa municipio, departamento y semestre. `geo_history` sustituye la identidad municipal por coordenadas y rezagos, y conserva departamento y semestre.

In [ ]:
GRAFICAR_CATEGORIAS = True

if GRAFICAR_CATEGORIAS:
    categorias = sorted({column for columns in EMBEDDING_COLUMNS.values() for column in columns})
    for column in categorias:
        figure = plot_categorical_encoding(dataset, column)
        display(figure)
        plt.close(figure)

## Pronósticos 2026 por municipio

Las barras representan el intervalo empírico basado en el percentil 90 del error absoluto observado durante el backtesting.

In [ ]:
display(pronostico)
figure = plot_forecast_panels(pronostico)
display(figure)
plt.close(figure)